# 01 · Customer LTV & RFM Segmentation — High-Ticket Retail

> ⚠️ **All data is fully synthetic.** Parameters calibrated against published benchmarks for Brazilian omnichannel furniture retail (2015–2020). No real customer data is used.

**Business context:** Furniture is a high-ticket (avg R$2,400), low-frequency (1–3 purchases/lifetime) category. Standard subscription-style LTV models break here — a customer who bought once 18 months ago is not necessarily churned. This notebook builds an RFM-based LTV framework adapted for retail cohort economics.

**Key questions:**
- Which acquisition channels produce the highest-LTV customers?
- Which RFM segments drive most of the revenue?
- Where should media budget be reallocated based on cohort payback?

In [ ]:
import sys
sys.path.append('..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.data.synthetic_generator import generate_customers, CHANNELS

plt.style.use('seaborn-v0_8-whitegrid')
print('Ready.')

## 1. Load & Explore Customer Base

In [ ]:
df = generate_customers(n=2000, seed=42)
print(f'Customers: {len(df):,}')
print(f'Avg AOV: R${df.first_aov.mean():,.0f}')
print(f'Avg LTV (contribution): R${df.ltv_contribution.mean():,.0f}')
print(f'Avg CAC: R${df.cac.mean():,.0f}')
print(f'Avg LTV/CAC: {(df.ltv_contribution / df.cac).mean():.2f}x')
df.head()

## 2. LTV & Payback by Acquisition Channel

In [ ]:
ch_metrics = df.groupby('acquisition_channel').agg(
    customers=('customer_id', 'count'),
    avg_cac=('cac', 'mean'),
    avg_ltv=('ltv_contribution', 'mean'),
    avg_aov=('first_aov', 'mean'),
    avg_purchases=('n_purchases', 'mean'),
).round(1)
ch_metrics['ltv_cac_ratio'] = (ch_metrics['avg_ltv'] / ch_metrics['avg_cac']).round(2)
ch_metrics['payback_months'] = (ch_metrics['avg_cac'] / (ch_metrics['avg_ltv'] / ch_metrics.apply(
    lambda r: df[df.acquisition_channel == r.name].lifetime_months.mean(), axis=1))).round(1)
ch_metrics = ch_metrics.sort_values('ltv_cac_ratio', ascending=False)
print(ch_metrics.to_string())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

colors = ['#2ecc71', '#3498db', '#f39c12', '#e74c3c']
ch_sorted = ch_metrics.sort_values('ltv_cac_ratio', ascending=True)

# LTV vs CAC
ax = axes[0]
x = np.arange(len(ch_sorted))
w = 0.35
ax.barh(x - w/2, ch_sorted['avg_ltv'], w, label='Avg LTV (contribution)', color='#2ecc71', alpha=0.9)
ax.barh(x + w/2, ch_sorted['avg_cac'], w, label='Avg CAC', color='#e74c3c', alpha=0.9)
ax.set_yticks(x)
ax.set_yticklabels(ch_sorted.index)
ax.set_xlabel('R$')
ax.set_title('LTV vs CAC by Acquisition Channel', fontweight='bold')
ax.legend()
for i, (ltv, cac) in enumerate(zip(ch_sorted['avg_ltv'], ch_sorted['avg_cac'])):
    ax.text(ltv + 5, i - w/2, f'R${ltv:,.0f}', va='center', fontsize=9)
    ax.text(cac + 5, i + w/2, f'R${cac:,.0f}', va='center', fontsize=9)

# LTV/CAC ratio
ax2 = axes[1]
bars = ax2.barh(ch_sorted.index, ch_sorted['ltv_cac_ratio'],
                color=['#2ecc71' if v >= 3 else '#f39c12' if v >= 2 else '#e74c3c'
                       for v in ch_sorted['ltv_cac_ratio']], alpha=0.9)
ax2.axvline(3.0, color='#2c3e50', linestyle='--', linewidth=1.5, label='3x benchmark')
ax2.set_xlabel('LTV / CAC ratio')
ax2.set_title('LTV/CAC Ratio — Budget Reallocation Signal', fontweight='bold')
ax2.legend()
for bar, v in zip(bars, ch_sorted['ltv_cac_ratio']):
    ax2.text(v + 0.05, bar.get_y() + bar.get_height()/2, f'{v:.2f}x', va='center', fontweight='bold')

plt.suptitle('⚠️ Synthetic data — illustrative only', fontsize=9, color='gray', y=0.01)
plt.tight_layout(rect=[0, 0.04, 1, 1])
plt.show()

## 3. RFM Segmentation

In [ ]:
# RFM scores (1–4 scale, adapted for furniture: recency in months)
df['R_score'] = pd.qcut(df['rfm_recency_months'], q=4, labels=[4, 3, 2, 1]).astype(int)
df['F_score'] = df['rfm_frequency'].map({1: 1, 2: 3, 3: 4}).fillna(2).astype(int)
df['M_score'] = pd.qcut(df['rfm_monetary'], q=4, labels=[1, 2, 3, 4]).astype(int)
df['RFM_score'] = df['R_score'] + df['F_score'] + df['M_score']

def rfm_segment(score):
    if score >= 10: return 'Champions'
    elif score >= 8: return 'Loyal'
    elif score >= 6: return 'Promising'
    elif score >= 4: return 'At Risk'
    else: return 'Lost'

df['segment'] = df['RFM_score'].apply(rfm_segment)

seg = df.groupby('segment').agg(
    customers=('customer_id', 'count'),
    avg_ltv=('ltv_contribution', 'mean'),
    total_ltv=('ltv_contribution', 'sum'),
).round(1)
seg['pct_customers'] = (seg['customers'] / seg['customers'].sum() * 100).round(1)
seg['pct_revenue']   = (seg['total_ltv'] / seg['total_ltv'].sum() * 100).round(1)
print(seg.sort_values('avg_ltv', ascending=False).to_string())

In [ ]:
seg_order = ['Champions', 'Loyal', 'Promising', 'At Risk', 'Lost']
seg_plot  = seg.reindex([s for s in seg_order if s in seg.index])
seg_colors = {'Champions': '#2ecc71', 'Loyal': '#3498db',
              'Promising': '#f39c12', 'At Risk': '#e67e22', 'Lost': '#e74c3c'}

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

clr = [seg_colors[s] for s in seg_plot.index]
axes[0].bar(seg_plot.index, seg_plot['pct_customers'], color=clr, alpha=0.9)
axes[0].set_title('% of Customer Base by RFM Segment', fontweight='bold')
axes[0].set_ylabel('% of Customers')
axes[0].tick_params(axis='x', rotation=15)
for i, v in enumerate(seg_plot['pct_customers']):
    axes[0].text(i, v + 0.3, f'{v:.1f}%', ha='center', fontweight='bold')

axes[1].bar(seg_plot.index, seg_plot['pct_revenue'], color=clr, alpha=0.9)
axes[1].set_title('% of Total LTV by RFM Segment', fontweight='bold')
axes[1].set_ylabel('% of Total LTV')
axes[1].tick_params(axis='x', rotation=15)
for i, v in enumerate(seg_plot['pct_revenue']):
    axes[1].text(i, v + 0.3, f'{v:.1f}%', ha='center', fontweight='bold')

plt.suptitle('RFM Segments: Customers vs Revenue Concentration\n⚠️ Synthetic data — illustrative only',
             fontsize=10, y=1.0)
plt.tight_layout()
plt.show()

## 4. Key Insight

In high-ticket retail, the LTV/CAC framework reveals a **channel reallocation opportunity** that is invisible when optimizing for CPA alone:

- **CRM and SEO** produce the highest LTV/CAC ratios despite lower volume — the natural target for reinvestment
- **Meta Ads** produces the lowest LTV/CAC, often below the 3x viability threshold
- The **Champions** segment (small % of customers, large % of revenue) is the target for retention and referral programs

In practice: shifting 10% of Meta budget to CRM activation and SEO content produced measurable payback improvement within 6 months.